In [0]:
# =============================================================================
# ICEBASE — PHASE 3 | NOTEBOOK 01
# Gold Layer — customer_360
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# STANDALONE NOTEBOOK — Run this attached to the icebase-dev cluster.
# This is NOT a pipeline source file. It is a regular Python notebook
# that reads from Silver and writes aggregated Gold tables using standard
# PySpark DataFrame writes.
#
# WHAT THIS NOTEBOOK DOES:
#   Builds the customer_360 Gold table — one row per fan, summarizing their
#   entire relationship with the Idaho Mashers. This is the single most
#   important table in the Gold layer. It feeds:
#     - Fan segmentation (Phase 4 K-Means model)
#     - Churn prediction (Phase 4 XGBoost model)
#     - Fan Health Dashboard (Phase 5)
#     - CMO-level business questions
#
# TABLE WRITTEN:
#   icebase.gold.customer_360
#
# RUN ORDER IN LAKEFLOW JOB (Phase 3 orchestration):
#   Task 1: Silver pipeline runs (icebase-bronze-to-silver)
#   Task 2: THIS NOTEBOOK (depends on Task 1)
#   Task 3: game_revenue notebook (runs in parallel with this one)
#   Task 4: retention_cohort notebook (depends on Tasks 2 & 3)
# =============================================================================

In [0]:
# COMMAND ----------
# ── CELL 1: Imports and Config ─────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
CATALOG = "icebase"
GOLD    = f"{CATALOG}.gold"
SILVER  = f"{CATALOG}.silver"
 
print("✅ customer_360 notebook loaded")
print(f"   Writing to: {GOLD}.customer_360")

In [0]:
# COMMAND ----------
# ── CELL 2: Build customer_360 ─────────────────────────────────────────────
#
# BUSINESS LOGIC:
#   One row per customer. Every column answers a CMO question:
#
#   total_spend          → "Who are our highest value fans?"
#   games_attended       → "Who is actually showing up?"
#   days_since_last      → "Who is drifting away right now?"
#   promo_sensitivity    → "Who only buys on discount?" (the bad story metric)
#   preferred_section    → "Where do our best fans sit?"
#   is_season_holder     → "Are they in our most loyal tier?"
#   jersey_night_attendee→ "Did they come to the biggest game of the year?"
#   hot_start_buyer      → "Did we retain fans from when we were winning?"
#
# RFM FEATURES (used directly by ML models in Phase 4):
#   recency  = days_since_last (lower = more recent = better)
#   frequency= games_attended
#   monetary = total_spend
 
fact   = spark.table(f"{SILVER}.fact_tickets")
dim_c  = spark.table(f"{SILVER}.dim_customer")
dim_g  = spark.table(f"{SILVER}.dim_game")
promo  = spark.table(f"{SILVER}.bridge_promo")
 
# ── Per-customer ticket aggregations ──────────────────────────────────────
ticket_agg = (
    fact.alias("f")
    .join(dim_g.select("game_id", "game_date", "season_phase").alias("g"), "game_id", "left")
    .groupBy("customer_id")
    .agg(
        # Spend metrics
        F.round(F.sum("ticket_price"), 2)           .alias("total_spend"),
        F.round(F.avg("ticket_price"), 2)           .alias("avg_ticket_price"),
        F.round(F.max("ticket_price"), 2)           .alias("max_ticket_price"),
 
        # Attendance metrics
        F.countDistinct("game_id")                  .alias("games_attended"),
        F.count("ticket_id")                        .alias("total_tickets"),
 
        # Recency — days since most recent game attended
        F.datediff(
            F.current_date(),
            F.max(F.col("f.game_date").cast("date"))
        )                                           .alias("days_since_last"),
 
        # Promo behavior
        F.sum(F.when(F.col("is_promo_ticket"), 1).otherwise(0))
                                                    .alias("promo_tickets"),
        F.round(
            F.sum(F.when(F.col("is_promo_ticket"), 1).otherwise(0))
            / F.count("ticket_id"), 4
        )                                           .alias("promo_rate"),
 
        # Purchase timing behavior
        F.sum(F.when(F.col("is_advance_purchase"), 1).otherwise(0))
                                                    .alias("advance_purchases"),
        F.round(F.avg("days_before_game"), 1)      .alias("avg_days_before_game"),
 
        # Section preference — most common tier purchased
        F.mode("section_tier")                     .alias("preferred_section"),
        F.round(F.avg("seat_tier_rank"), 2)        .alias("avg_seat_tier_rank"),
 
        # Special event flags
        F.max(F.when(F.col("is_jersey_night_game"), 1).otherwise(0)).cast("boolean")
                                                    .alias("jersey_night_attendee"),
        F.max(F.when(F.col("is_lapsed_reactivation"), 1).otherwise(0)).cast("boolean")
                                                    .alias("was_lapsed_reactivation"),
 
        # Phase attendance — did they buy during each story arc?
        F.max(F.when(F.col("g.season_phase") == "hot_start", 1).otherwise(0)).cast("boolean")
                                                    .alias("hot_start_buyer"),
        F.max(F.when(F.col("g.season_phase") == "slump", 1).otherwise(0)).cast("boolean")
                                                    .alias("slump_buyer"),
        F.max(F.when(F.col("g.season_phase") == "late_push", 1).otherwise(0)).cast("boolean")
                                                    .alias("late_push_buyer"),
 
        # Channel diversity — how many different channels used?
        F.countDistinct("purchase_channel")        .alias("channels_used"),
    )
)
 
# ── Promo spend from bridge_promo ─────────────────────────────────────────
promo_agg = (
    promo
    .groupBy("customer_id")
    .agg(
        F.round(F.sum("discount_amount"), 2)        .alias("total_discount_received"),
        F.round(F.avg("promo_impact_score"), 2)     .alias("avg_promo_impact_score"),
        F.mode("discount_tier")                     .alias("most_common_discount_tier"),
    )
)
 
# ── Join everything together ──────────────────────────────────────────────
customer_360 = (
    dim_c
    .select(
        "customer_id", "full_name", "email", "zip_code",
        "acquisition_channel", "initial_segment", "is_season_holder",
        "signup_date", "tenure_days", "is_jersey_night_acq",
        "is_deeply_lapsed", "record_source"
    )
    .join(ticket_agg, "customer_id", "left")
    .join(promo_agg,  "customer_id", "left")
    .withColumn("total_discount_received",
        F.coalesce(F.col("total_discount_received"), F.lit(0.0)))
    .withColumn("promo_tickets",
        F.coalesce(F.col("promo_tickets"), F.lit(0)))
    .withColumn("promo_rate",
        F.coalesce(F.col("promo_rate"), F.lit(0.0)))
    # Promo sensitivity score: 0.0–1.0 — high = only buys with discounts
    # Blend of promo_rate and discount depth
    .withColumn("promo_sensitivity",
        F.round(
            F.coalesce(F.col("promo_rate"), F.lit(0.0)) * 0.7 +
            F.when(F.col("most_common_discount_tier") == "deep",   0.3)
             .when(F.col("most_common_discount_tier") == "moderate", 0.15)
             .otherwise(0.0),
            4
        )
    )
    # Churn risk flag: no purchase in 45+ days = at risk
    .withColumn("is_churn_risk",
        F.when(
            F.col("days_since_last") > 45, True
        ).otherwise(False)
    )
    # Revenue net = total spend minus discounts received
    .withColumn("revenue_net",
        F.round(
            F.coalesce(F.col("total_spend"), F.lit(0.0)) -
            F.coalesce(F.col("total_discount_received"), F.lit(0.0)),
            2
        )
    )
    .withColumn("_gold_refreshed_at", F.current_timestamp())
)

In [0]:
# COMMAND ----------
# ── CELL 3: Write to Gold ──────────────────────────────────────────────────
 
(
    customer_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.customer_360")
)
 
count = spark.table(f"{GOLD}.customer_360").count()
print(f"✅ customer_360 written | {count:,} rows")

In [0]:
# COMMAND ----------
# ── CELL 4: Quick Sanity Check ─────────────────────────────────────────────
 
print("── Segment breakdown by initial_segment ──")
display(
    spark.table(f"{GOLD}.customer_360")
    .groupBy("initial_segment")
    .agg(
        F.count("*")                            .alias("fans"),
        F.round(F.avg("total_spend"), 2)        .alias("avg_spend"),
        F.round(F.avg("games_attended"), 1)     .alias("avg_games"),
        F.round(F.avg("promo_sensitivity"), 3)  .alias("avg_promo_sensitivity"),
        F.round(F.avg("days_since_last"), 0)    .alias("avg_days_since_last"),
        F.sum(F.col("is_churn_risk").cast("int")).alias("churn_risk_count"),
    )
    .orderBy(F.col("avg_spend").desc())
)
 
# EXPECTED PATTERNS:
#   season_core      → highest avg_spend, lowest promo_sensitivity, lowest days_since_last
#   promo_hunter     → highest promo_sensitivity (>0.7)
#   deeply_lapsed    → highest days_since_last, is_churn_risk = TRUE for nearly all
#   high_value_new   → moderate spend, low promo_sensitivity (early in lifecycle)